# Decision Tree Regressor

A Decision Tree is one of the most intuitive supervised learning algorithms. It builds a tree-like structure where each internal node represents a decision based on a feature, each branch represents the outcome of that decision, and each leaf node holds a predicted value. For regression tasks, the tree learns to split the data recursively to minimize prediction error, ultimately assigning the mean of the target values at each leaf as the prediction.

## 1. Key Concepts

A Decision Tree Regressor is a **parametric, eager learner** — unlike KNN, it builds an explicit model during training. The tree is constructed by recursively partitioning the training data $\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^N$ into subsets that are increasingly homogeneous with respect to the target variable $y$.

At each node, the algorithm selects the best **(feature, threshold)** pair that maximizes the reduction of variance in the target values. The process continues until a stopping criterion is met (e.g., maximum depth, minimum samples per node), at which point the node becomes a **leaf** and returns the mean of its samples as the prediction:

$$\hat{y} = \frac{1}{|\mathcal{S}|} \sum_{i \in \mathcal{S}} y_i$$

Where $\mathcal{S}$ is the set of sample indices at the leaf node.

## 2. Splitting Criterion

To determine the best split at each node, the algorithm evaluates all possible **(feature, threshold)** combinations and selects the one that maximizes **variance reduction** (also called impurity decrease).

### Mean Squared Error (MSE) at a node

The impurity of a node is measured using the MSE of the target values it contains:

$$\text{MSE}(\mathcal{S}) = \frac{1}{|\mathcal{S}|} \sum_{i \in \mathcal{S}} (y_i - \bar{y})^2$$

Where $\bar{y}$ is the mean of the target values in the node.

### Variance Reduction

For a split that partitions a node $\mathcal{S}$ into a left child $\mathcal{S}_L$ and a right child $\mathcal{S}_R$, the variance reduction is:

$$\Delta = \text{MSE}(\mathcal{S}) - \left( \frac{|\mathcal{S}_L|}{|\mathcal{S}|} \cdot \text{MSE}(\mathcal{S}_L) + \frac{|\mathcal{S}_R|}{|\mathcal{S}|} \cdot \text{MSE}(\mathcal{S}_R) \right)$$

The split with the highest $\Delta$ is selected. A split is only performed if $\Delta \geq$ `min_impurity_decrease`.

**Note**: The implementation uses a candidate threshold strategy for efficiency. If a feature has more than 10 unique values, only the 25th, 50th, and 75th percentiles are evaluated as candidate thresholds, reducing the number of splits to consider without significantly sacrificing accuracy.

## 3. Stopping Criteria

The tree stops growing a branch (i.e., creates a leaf node) when **any** of the following conditions is met:

| Condition | Parameter | Description |
|-----------|-----------|-------------|
| Too few samples to split | `min_samples_split` | Node has fewer samples than the threshold |
| Maximum depth reached | `max_depth` | Tree has reached the allowed depth |
| Pure node | — | All target values in the node are identical |
| Insufficient gain | `min_impurity_decrease` | No split achieves the required variance reduction |
| Leaf too small | `min_samples_leaf` | A split would produce a child with too few samples |

## 4. Pseudo-algorithm

**Build phase** — `fit(X, y, depth)`:

1. $\mathcal{D} \leftarrow$ training set of $N$ samples $(\mathbf{x}_i, y_i)$
2. **if** stopping criterion met **then** **return** $\text{leaf}(\bar{y})$
3. **for each** feature $j$ and candidate threshold $t$ **do**
4. $\quad \mathcal{S}_L \leftarrow \{i : x_{ij} < t\}$, $\mathcal{S}_R \leftarrow \{i : x_{ij} \geq t\}$
5. $\quad \Delta_{j,t} \leftarrow \text{VarianceReduction}(\mathcal{S}, \mathcal{S}_L, \mathcal{S}_R)$
6. **end for**
7. $(j^*, t^*) \leftarrow \arg\max_{j,t} \; \Delta_{j,t}$
8. **if** $\Delta_{j^*, t^*} < $ `min_impurity_decrease` **then return** $\text{leaf}(\bar{y})$
9. **return** Node($j^*$, $t^*$, left=`fit`($\mathcal{S}_L$, depth+1), right=`fit`($\mathcal{S}_R$, depth+1))

**Predict phase** — `predict(x)`:

10. **if** node is a leaf **then return** $\hat{y}$ (leaf value)
11. **if** $x[j^*] < t^*$ **then** traverse left child
12. **else** traverse right child
13. **return** prediction at reached leaf

## 5. Implementation

For this implementation, we use the **California Housing** dataset, a standard regression benchmark. It contains 20,640 samples representing California census blocks, with 8 numerical features (median income, house age, average rooms, etc.) and a continuous target: the **median house value** (in $100,000s).

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_california_housing
from ifri_mini_ml_lib.preprocessing.preparation import DataSplitter

housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = pd.Series(housing.target)

In [ ]:
X.head()

In [ ]:
# Split the data into training and testing sets
splitter = DataSplitter(seed=42)
X_train, X_test, y_train, y_test = splitter.train_test_split(X, y, test_size=0.2)

### With Scikit-learn

In [ ]:
import time

In [ ]:
from sklearn.tree import DecisionTreeRegressor as SklearnDTR

start_1 = time.perf_counter()
sk_tree = SklearnDTR(max_depth=5, min_samples_split=2, min_samples_leaf=1)
sk_tree.fit(X_train, y_train)
end_1 = time.perf_counter()

### With ifri-mini-ml-lib

In [ ]:
from ifri_mini_ml_lib.regression import DecisionTreeRegressor

start_2 = time.perf_counter()
my_tree = DecisionTreeRegressor(max_depth=5, min_samples_split=2, min_samples_leaf=1)
my_tree.fit(X_train.values, y_train.values)
end_2 = time.perf_counter()

In [ ]:
# Predict on the test set
y_pred_1 = sk_tree.predict(X_test)
y_pred_2 = my_tree.predict(X_test.values)

In [ ]:
# Evaluate both models
from ifri_mini_ml_lib.metrics.regression import mean_squared_error, mean_absolute_error, r2_score

mse_1   = mean_squared_error(y_test, y_pred_1)
mse_2   = mean_squared_error(y_test, y_pred_2)

mae_1   = mean_absolute_error(y_test, y_pred_1)
mae_2   = mean_absolute_error(y_test, y_pred_2)

r2_1    = r2_score(y_test, y_pred_1)
r2_2    = r2_score(y_test, y_pred_2)

In [ ]:
# Summary table
results = pd.DataFrame({
    'Metric': ['MSE', 'MAE', 'R² Score', 'Training Time (s)'],
    'Scikit-learn': [mse_1, mae_1, r2_1, end_1 - start_1],
    'ifri-mini-ml-lib': [mse_2, mae_2, r2_2, end_2 - start_2],
})

results.T

## 6. Interactive Demo

In [ ]:
from ipywidgets import interact, IntSlider, FloatSlider
from notebooks.regression.utils import plot_decision_tree_regressor

interact(
    plot_decision_tree_regressor,
    max_depth=IntSlider(min=1, max=10, step=1, value=3, description='Max Depth:'),
    min_samples_split=IntSlider(min=2, max=20, step=1, value=2, description='Min Split:'),
    min_samples_leaf=IntSlider(min=1, max=20, step=1, value=1, description='Min Leaf:'),
    library=__import__('ipywidgets').Dropdown(
        options=['sklearn', 'ifri_mini_ml_lib'],
        value='sklearn',
        description='Library:'
    )
);

The interactive demo above lets you explore how the tree's hyperparameters affect its predictions. Increasing `max_depth` allows the tree to capture more complex patterns — but also risks **overfitting** the training data. Increasing `min_samples_split` and `min_samples_leaf` constrains tree growth and acts as a regularization mechanism.

## 7. Real-life Applications

Decision Tree Regressors are widely adopted across industries thanks to their interpretability and ability to model non-linear relationships without feature scaling.

1. **Real Estate Pricing**: Decision trees are used to estimate property values based on features such as location, size, number of rooms, and proximity to amenities. Their interpretable splits make them easy to explain to stakeholders.

2. **Energy Forecasting**: Utilities and smart grid operators use decision trees to predict energy consumption and production based on weather conditions, time of day, and historical usage patterns.

3. **Medical Outcome Prediction**: In clinical settings, decision trees help predict continuous outcomes such as patient recovery time, medication dosage, or biomarker levels — and their transparent rules support medical auditing.

4. **Financial Risk Assessment**: Banks and insurers use decision trees to estimate credit risk scores, loan default probabilities, or claim amounts, leveraging their capacity to handle mixed feature types without preprocessing.

Decision trees also serve as the **base learner** in powerful ensemble methods such as **Random Forests** and **Gradient Boosting Machines**, which build on their strengths while mitigating overfitting.

## 8. Limitations and Challenges

Despite their intuitive appeal, decision tree regressors have notable weaknesses to keep in mind.

1. **Overfitting**: Without constraints, a decision tree can grow deep enough to memorize the training set entirely — achieving near-zero training error while performing poorly on unseen data. Regularization through `max_depth`, `min_samples_leaf`, and `min_impurity_decrease` is essential.

2. **High Variance (Instability)**: Small changes in the training data can lead to very different tree structures. A single tree is highly sensitive to noise and outliers, which makes it an unstable predictor.

3. **Axis-Aligned Splits Only**: The splitting criterion always partitions the feature space with axis-parallel hyperplanes. This means that diagonal or curved decision boundaries require a large number of splits to approximate, making the model inefficient for such data distributions.

4. **Greedy Construction**: The tree is built using a greedy strategy — each split is locally optimal but not globally. There is no guarantee that the resulting tree minimizes the overall prediction error.

These limitations motivate the use of ensemble methods (Random Forest, Gradient Boosting) that combine many trees to reduce variance and improve generalization.

## 9. References

- Decision Trees, Scikit-learn User Guide, https://scikit-learn.org/stable/modules/tree.html
- Breiman, L. et al. (1984). *Classification and Regression Trees (CART)*. Wadsworth.
- Decision Tree Regression, Towards Data Science, https://towardsdatascience.com/decision-tree-regression-explained/
- Understanding Decision Trees, StatQuest with Josh Starmer, https://www.youtube.com/watch?v=g9c66TUylZ4